# Reproduced v2/v2.1 vs original score correlations

This notebook is a runnable wrapper around `scripts/correlate_original_reproduced.py`. Running all cells regenerates the correlation CSVs from `data/reproduced_v2`, `data/reproduced_v2.1`, and `data/original/scores/firmquarter_2022q1.csv`, then displays the combined outputs.

The reproduced measures use raw numerators divided by `nr_of_words`: `risk / nr_of_words` for PRisk and PRiskT, and `sentiment / nr_of_words` for PSentiment.


In [1]:
from __future__ import annotations

import csv
import math
import subprocess
import sys
from html import escape
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "scripts" / "correlate_original_reproduced.py").exists():
            return path
    raise RuntimeError("Could not find the prisk-replication repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
SCRIPT = REPO_ROOT / "scripts" / "correlate_original_reproduced.py"
CORRELATIONS_DIR = REPO_ROOT / "data" / "correlations"

REPO_ROOT


PosixPath('/Users/markusschwedeler/Projects/prisk-replication')

## Regenerate Correlations


In [2]:
result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()


Wrote data/correlations/reproduced_v2_gvkey_dateq_correlations.csv
Wrote data/correlations/reproduced_v2_dateq_correlations.csv
Wrote data/correlations/reproduced_v2.1_gvkey_dateq_correlations.csv
Wrote data/correlations/reproduced_v2.1_dateq_correlations.csv
Wrote data/correlations/reproduced_versions_gvkey_dateq_correlations.csv
Wrote data/correlations/reproduced_versions_dateq_correlations.csv



In [3]:
def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(newline="") as file:
        return list(csv.DictReader(file))


def as_number(value: str) -> float | None:
    try:
        parsed = float(value)
    except (TypeError, ValueError):
        return None
    if not math.isfinite(parsed):
        return None
    return parsed


def format_value(value: object) -> str:
    if value is None:
        return ""
    if isinstance(value, float):
        return f"{value:.6f}"
    text = str(value)
    parsed = as_number(text)
    if parsed is not None and "." in text:
        return f"{parsed:.6f}"
    return text


def display_table(rows: list[dict[str, object]]):
    try:
        from IPython.display import HTML
    except Exception:
        return rows

    if not rows:
        return HTML("<em>No rows</em>")

    columns = list(rows[0].keys())
    header = "".join(f"<th>{escape(column)}</th>" for column in columns)
    body_rows = []
    for row in rows:
        cells = "".join(
            f"<td>{escape(format_value(row.get(column)))}</td>"
            for column in columns
        )
        body_rows.append(f"<tr>{cells}</tr>")
    return HTML(
        "<table>"
        "<thead><tr>" + header + "</tr></thead>"
        "<tbody>" + "".join(body_rows) + "</tbody>"
        "</table>"
    )


## Generated Files


In [4]:
generated_files = [
    "reproduced_v2_gvkey_dateq_correlations.csv",
    "reproduced_v2_dateq_correlations.csv",
    "reproduced_v2.1_gvkey_dateq_correlations.csv",
    "reproduced_v2.1_dateq_correlations.csv",
    "reproduced_versions_gvkey_dateq_correlations.csv",
    "reproduced_versions_dateq_correlations.csv",
]

display_table(
    [
        {
            "file": name,
            "exists": (CORRELATIONS_DIR / name).exists(),
            "size_bytes": (CORRELATIONS_DIR / name).stat().st_size if (CORRELATIONS_DIR / name).exists() else "",
        }
        for name in generated_files
    ]
)


file,exists,size_bytes
reproduced_v2_gvkey_dateq_correlations.csv,True,1876
reproduced_v2_dateq_correlations.csv,True,1830
reproduced_v2.1_gvkey_dateq_correlations.csv,True,1888
reproduced_v2.1_dateq_correlations.csv,True,1840
reproduced_versions_gvkey_dateq_correlations.csv,True,3973
reproduced_versions_dateq_correlations.csv,True,3867


## Combined gvkey-dateQ Correlations


In [5]:
gvkey_dateq_correlations = read_csv(
    CORRELATIONS_DIR / "reproduced_versions_gvkey_dateq_correlations.csv"
)
display_table(gvkey_dateq_correlations)


reproduced_version,series,original_column,reproduced_value,reproduced_file,first_dateQ,last_dateQ,level,n_pairs,n_dateQ,pearson
reproduced_v2,PRisk,PRisk,risk / nr_of_words,NLA_2026-06-22T161036_prisk-and-psentiment-v2_67e91380_firmlevel.csv,2002q1,2022q1,gvkey-dateQ,310303,81,0.904326
reproduced_v2,PSentiment,PSentiment,sentiment / nr_of_words,NLA_2026-06-22T161036_prisk-and-psentiment-v2_67e91380_firmlevel.csv,2002q1,2022q1,gvkey-dateQ,310303,81,0.855194
reproduced_v2,PRiskT_economic,PRiskT_economic,risk / nr_of_words,NLA_2026-06-22T180907_topic-based-prisk-economy-v2_48b5c0a6_firmlevel.csv,2002q1,2022q1,gvkey-dateQ,310303,81,0.908175
reproduced_v2,PRiskT_environment,PRiskT_environment,risk / nr_of_words,NLA_2026-06-22T165155_topic-based-prisk-environment-v2_9de50ab6_firmlevel.csv,2002q1,2022q1,gvkey-dateQ,310303,81,0.858467
reproduced_v2,PRiskT_trade,PRiskT_trade,risk / nr_of_words,NLA_2026-06-22T172546_topic-based-prisk-trade-v2_45a8c11c_firmlevel.csv,2002q1,2022q1,gvkey-dateQ,310303,81,0.806010
reproduced_v2,PRiskT_institutions,PRiskT_institutions,risk / nr_of_words,NLA_2026-06-22T191432_topic-based-prisk-institutions-v2_6687e12f_firmlevel.csv,2002q1,2022q1,gvkey-dateQ,310303,81,0.893494
reproduced_v2,PRiskT_health,PRiskT_health,risk / nr_of_words,NLA_2026-06-22T191432_topic-based-prisk-health-v2_f46cf5ed_firmlevel.csv,2002q1,2022q1,gvkey-dateQ,310303,81,0.850774
reproduced_v2,PRiskT_security,PRiskT_security,risk / nr_of_words,NLA_2026-06-22T185843_topic-based-prisk-security-v2_38ed823b_firmlevel.csv,2002q1,2022q1,gvkey-dateQ,310303,81,0.815520
reproduced_v2,PRiskT_tax,PRiskT_tax,risk / nr_of_words,NLA_2026-06-22T172005_topic-based-prisk-tax-v2_241dc566_firmlevel.csv,2002q1,2022q1,gvkey-dateQ,310303,81,0.903212
reproduced_v2,PRiskT_technology,PRiskT_technology,risk / nr_of_words,NLA_2026-06-22T163350_topic-based-prisk-technology-v2_f70b42f9_firmlevel.csv,2002q1,2022q1,gvkey-dateQ,310303,81,0.868482


## Combined dateQ Correlations


In [6]:
dateq_correlations = read_csv(
    CORRELATIONS_DIR / "reproduced_versions_dateq_correlations.csv"
)
display_table(dateq_correlations)


reproduced_version,series,original_column,reproduced_value,reproduced_file,first_dateQ,last_dateQ,level,n_pairs,matched_gvkey_dateQ,pearson
reproduced_v2,PRisk,PRisk,risk / nr_of_words,NLA_2026-06-22T161036_prisk-and-psentiment-v2_67e91380_firmlevel.csv,2002q1,2022q1,dateQ,81,310303,0.981339
reproduced_v2,PSentiment,PSentiment,sentiment / nr_of_words,NLA_2026-06-22T161036_prisk-and-psentiment-v2_67e91380_firmlevel.csv,2002q1,2022q1,dateQ,81,310303,0.945431
reproduced_v2,PRiskT_economic,PRiskT_economic,risk / nr_of_words,NLA_2026-06-22T180907_topic-based-prisk-economy-v2_48b5c0a6_firmlevel.csv,2002q1,2022q1,dateQ,81,310303,0.991983
reproduced_v2,PRiskT_environment,PRiskT_environment,risk / nr_of_words,NLA_2026-06-22T165155_topic-based-prisk-environment-v2_9de50ab6_firmlevel.csv,2002q1,2022q1,dateQ,81,310303,0.982627
reproduced_v2,PRiskT_trade,PRiskT_trade,risk / nr_of_words,NLA_2026-06-22T172546_topic-based-prisk-trade-v2_45a8c11c_firmlevel.csv,2002q1,2022q1,dateQ,81,310303,0.953507
reproduced_v2,PRiskT_institutions,PRiskT_institutions,risk / nr_of_words,NLA_2026-06-22T191432_topic-based-prisk-institutions-v2_6687e12f_firmlevel.csv,2002q1,2022q1,dateQ,81,310303,0.985902
reproduced_v2,PRiskT_health,PRiskT_health,risk / nr_of_words,NLA_2026-06-22T191432_topic-based-prisk-health-v2_f46cf5ed_firmlevel.csv,2002q1,2022q1,dateQ,81,310303,0.981758
reproduced_v2,PRiskT_security,PRiskT_security,risk / nr_of_words,NLA_2026-06-22T185843_topic-based-prisk-security-v2_38ed823b_firmlevel.csv,2002q1,2022q1,dateQ,81,310303,0.966916
reproduced_v2,PRiskT_tax,PRiskT_tax,risk / nr_of_words,NLA_2026-06-22T172005_topic-based-prisk-tax-v2_241dc566_firmlevel.csv,2002q1,2022q1,dateQ,81,310303,0.989223
reproduced_v2,PRiskT_technology,PRiskT_technology,risk / nr_of_words,NLA_2026-06-22T163350_topic-based-prisk-technology-v2_f70b42f9_firmlevel.csv,2002q1,2022q1,dateQ,81,310303,0.985101


## v2.1 Minus v2


In [7]:
def rows_by_version_and_series(rows: list[dict[str, str]]) -> dict[tuple[str, str], dict[str, str]]:
    return {
        (row["reproduced_version"], row["series"]): row
        for row in rows
    }


gvkey_lookup = rows_by_version_and_series(gvkey_dateq_correlations)
dateq_lookup = rows_by_version_and_series(dateq_correlations)
series_names = [
    row["series"]
    for row in gvkey_dateq_correlations
    if row["reproduced_version"] == "reproduced_v2"
]

deltas = []
for series in series_names:
    gvkey_v2 = as_number(gvkey_lookup[("reproduced_v2", series)]["pearson"])
    gvkey_v21 = as_number(gvkey_lookup[("reproduced_v2.1", series)]["pearson"])
    dateq_v2 = as_number(dateq_lookup[("reproduced_v2", series)]["pearson"])
    dateq_v21 = as_number(dateq_lookup[("reproduced_v2.1", series)]["pearson"])
    deltas.append(
        {
            "series": series,
            "gvkey_dateQ_v2": gvkey_v2,
            "gvkey_dateQ_v2.1": gvkey_v21,
            "gvkey_dateQ_delta": None if gvkey_v2 is None or gvkey_v21 is None else gvkey_v21 - gvkey_v2,
            "dateQ_v2": dateq_v2,
            "dateQ_v2.1": dateq_v21,
            "dateQ_delta": None if dateq_v2 is None or dateq_v21 is None else dateq_v21 - dateq_v2,
        }
    )

display_table(deltas)


series,gvkey_dateQ_v2,gvkey_dateQ_v2.1,gvkey_dateQ_delta,dateQ_v2,dateQ_v2.1,dateQ_delta
PRisk,0.904326,0.905168,0.000842,0.981339,0.981506,0.000167
PSentiment,0.855194,0.856298,0.001104,0.945431,0.945560,0.000128
PRiskT_economic,0.908175,0.908175,0.000000,0.991983,0.991983,0.000000
PRiskT_environment,0.858467,0.858467,0.000000,0.982627,0.982627,0.000000
PRiskT_trade,0.806010,0.806010,0.000000,0.953507,0.953507,0.000000
PRiskT_institutions,0.893494,0.893494,0.000000,0.985902,0.985902,0.000000
PRiskT_health,0.850774,0.850774,0.000000,0.981758,0.981758,0.000000
PRiskT_security,0.815520,0.815520,0.000000,0.966916,0.966916,0.000000
PRiskT_tax,0.903212,0.903212,0.000000,0.989223,0.989223,0.000000
PRiskT_technology,0.868482,0.868482,0.000000,0.985101,0.985101,0.000000
